In [10]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE

from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, accuracy_score)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix,classification_report,f1_score,roc_auc_score

In [2]:
dataset= pd.read_csv('Digital_Marketing_DP.csv')
dataset.head()

,Age,Income,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,...,LoyaltyPoints,Conversion,Gender_Male,CampaignChannel_PPC,CampaignChannel_Referral,CampaignChannel_SEO,CampaignChannel_Social Media,CampaignType_Consideration,CampaignType_Conversion,CampaignType_Retention
0,56,136912,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,...,688,1,0,0,0,0,1,0,0,0
1,69,41760,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,...,3459,1,1,0,0,0,0,0,0,1
2,46,88456,1546.429596,0.277490,0.076423,2,8.223619,13.794901,0,11,...,2337,1,0,1,0,0,0,0,0,0
3,32,44085,539.525936,0.137611,0.088004,47,4.540939,14.688363,89,2,...,2463,1,0,1,0,0,0,0,1,0
4,60,83964,1678.043573,0.252851,0.109940,0,2.046847,13.993370,6,6,...,4345,1,0,1,0,0,0,0,1,0


In [8]:
cat_cols = dataset.select_dtypes(include=["object"]).columns.tolist()
print(f"\nEncoding categorical columns: {cat_cols}")

le = LabelEncoder()
for col in cat_cols:
    dataset[col] = le.fit_transform(dataset[col].astype(str))
X = dataset.drop("Conversion", axis=1)
y = dataset["Conversion"]

# Scale features (needed for SVM, LR, KNN; harmless for tree-based)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Also keep min-max scaled (non-negative) for Chi-Square
from sklearn.preprocessing import MinMaxScaler
X_minmax = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain size: {X_train.shape} | Test size: {X_test.shape}")




Encoding categorical columns: []

Train size: (6400, 21) | Test size: (1600, 21)


In [11]:
re=grid.cv_results_
grid_predictions = grid.predict(X_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
roc_score = roc_auc_score(y_test,grid.predict_proba(X_test)[:,1])

In [12]:
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("\nThe confusion Matrix:\n",cm)
print("\nThe report:\n",clf_report)
print("\nROC_AUC_Score:",roc_score)

The f1_macro value for best parameter {'criterion': 'entropy', 'max_features': 'sqrt', 'n_estimators': 100}: 0.8933522246107745

The confusion Matrix:
 [[  92  106]
 [  54 1348]]

The report:
               precision    recall  f1-score   support

           0       0.63      0.46      0.53       198
           1       0.93      0.96      0.94      1402

    accuracy                           0.90      1600
   macro avg       0.78      0.71      0.74      1600
weighted avg       0.89      0.90      0.89      1600


ROC_AUC_Score: 0.8011678842634621


In [13]:
#using pipeline to store scaler and best model.
from sklearn.pipeline import Pipeline
final_pipeline = Pipeline([
    ('scaler', scaler),
    ('model', grid.best_estimator_)   # best RandomForest model
])

In [14]:
import pickle
filename="Digital_Marketing_Dataset_GridRandomForest.pkl"
pickle.dump(final_pipeline, open(filename, 'wb'))